## Working Flow

`Document Loader`
➡️ `Text Splitter`
➡️ `Embeddings`
➡️ `Vector Store`
➡️ `Retriever`

## 1. TEXT SPLITTER

Why Text Splitting is Mandatory ?

- LLMs have:

    - Context length limits
    - Embeddings work best on small chunks

- One document ≠ one embedding
- So we split documents into semantic chunks.

##### A Text Splitter divides large documents into smaller, overlapping chunks so that embeddings capture context efficiently.

In [1]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('E:\\SSPL_Internship_Repo\\Jan_12_25\\RAG_tutorial\\data\\sample.txt')
documents = loader.load()

e:\SSPL_Internship_Repo\Jan_12_25\RAG_tutorial\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks[:2]): # Print first 2 chunks as a sample
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)

Number of chunks: 4

--- Chunk 1 ---
AI Agents: Overview and Applications

An AI agent is a software program designed to perceive its environment, make decisions, and take actions to achieve specific goals. These autonomous systems can operate independently or in collaboration with other agents.

--- Chunk 2 ---
Key Characteristics of AI Agents:
- Autonomy: Agents can operate without constant human intervention
- Reactivity: They respond to environmental changes in real-time
- Proactivity: Agents can take initiative to achieve their goals
- Social Ability: They can communicate and cooperate with other agents


## 2. Embeddings

- Embeddings convert text into numerical vectors so similarity search can be performed.

In RAG:
- Query → embedding
- Document chunks → embeddings
- Similarity search → relevant chunks

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Dell\AppData\Local\Temp\ipykernel_44172\2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
e:\SSPL_Internship_Repo\Jan_12_25\RAG_tutorial\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment vari

## 3. VECTOR STORE

- A Vector Store stores embeddings and allows fast similarity search.

Popular ones:
- FAISS (local)
- Chroma
- Pinecone (cloud)

In [11]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

## 4. RETRIVER

- A Retriever fetches the most relevant document chunks for a query.

In [12]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [14]:
docs = retriever.invoke(
    "What is this document about?"
)

print(f"Number of relevant documents retrieved: {len(docs)}")
if docs:
    print(docs[0].page_content)

Number of relevant documents retrieved: 3
AI Agents: Overview and Applications

An AI agent is a software program designed to perceive its environment, make decisions, and take actions to achieve specific goals. These autonomous systems can operate independently or in collaboration with other agents.
